# 05. Pandas 이해 및 활용 ③ DataFrame 병합하기

**AICE Associate 학습 5단계**

실무 데이터는 여러 파일(테이블)로 나뉘어 있는 경우가 많습니다. 예를 들어 통신사에서는
**고객 정보**와 **서비스 이용 정보**가 다른 테이블에 저장되어 있고, 이를 고객ID로 합쳐야 분석할 수 있습니다.

### 학습 목표
1. `pd.concat`으로 데이터를 위아래/좌우로 이어 붙일 수 있다.
2. `pd.merge`로 공통 키를 기준으로 결합할 수 있고, `how` 옵션의 차이를 이해한다.
3. 병합 후 행 수 변화와 결측치를 점검할 수 있다.

### 사용 데이터
- `cust_info.csv` : 고객 기본 정보 (customerID, gender, SeniorCitizen, Partner, Dependents, tenure, Churn)
- `service_info.csv` : 서비스 가입 정보 (customerID, PhoneService, InternetService, Contract, MonthlyCharges, TotalCharges 등)
- `cust_data.csv`, `cust_data1.csv` : 같은 구조의 고객 데이터

In [1]:
import pandas as pd
import numpy as np

---
## 1. 개념 익히기: 작은 예제

In [2]:
left = pd.DataFrame({'id': [1, 2, 3, 4], 'name': ['kim', 'lee', 'park', 'choi']})
right = pd.DataFrame({'id': [2, 3, 4, 5], 'plan': ['5G', 'LTE', '5G', 'LTE']})
display(left, right)

,id,name
0,1,kim
1,2,lee
2,3,park
3,4,choi


,id,plan
0,2,5G
1,3,LTE
2,4,5G
3,5,LTE


### 1-1. `pd.merge` : 공통 키로 결합 (SQL의 JOIN)

| how | 결과 |
|---|---|
| `'inner'` (기본값) | 양쪽에 **모두** 있는 키만 |
| `'left'` | 왼쪽 기준 (오른쪽에 없으면 NaN) |
| `'right'` | 오른쪽 기준 |
| `'outer'` | 양쪽 **전부** (없는 쪽은 NaN) |

In [3]:
pd.merge(left, right, on='id')                 # inner

,id,name,plan
0,2,lee,5G
1,3,park,LTE
2,4,choi,5G


In [4]:
pd.merge(left, right, on='id', how='left')

,id,name,plan
0,1,kim,NaN
1,2,lee,5G
2,3,park,LTE
3,4,choi,5G


In [5]:
pd.merge(left, right, on='id', how='outer', indicator=True)   # _merge: 어느 쪽에서 왔는지 표시

,id,name,plan,_merge
0,1,kim,NaN,left_only
1,2,lee,5G,both
2,3,park,LTE,both
3,4,choi,5G,both
4,5,NaN,LTE,right_only


In [6]:
# 키 컬럼 이름이 다를 때
right2 = right.rename(columns={'id': 'cust_no'})
pd.merge(left, right2, left_on='id', right_on='cust_no', how='inner')

,id,name,cust_no,plan
0,2,lee,2,5G
1,3,park,3,LTE
2,4,choi,4,5G


In [7]:
# DataFrame 메서드로도 가능
left.merge(right, on='id', how='left')

,id,name,plan
0,1,kim,NaN
1,2,lee,5G
2,3,park,LTE
3,4,choi,5G


### 1-2. `pd.concat` : 단순히 이어 붙이기

- `axis=0` (기본값): **위아래**로 (행 추가)
- `axis=1` : **좌우**로 (열 추가, 인덱스 기준으로 맞춤)

In [8]:
a = pd.DataFrame({'x': [1, 2], 'y': ['a', 'b']})
b = pd.DataFrame({'x': [3, 4], 'y': ['c', 'd']})
pd.concat([a, b])                          # 인덱스가 0,1,0,1 로 중복

,x,y
0,1,a
1,2,b
0,3,c
1,4,d


In [9]:
pd.concat([a, b], ignore_index=True)       # 인덱스 새로 매김

,x,y
0,1,a
1,2,b
2,3,c
3,4,d


In [10]:
pd.concat([a, b], axis=1)

,x,y,x,y
0,1,a,3,c
1,2,b,4,d


In [11]:
# 컬럼이 일부 다르면 없는 곳은 NaN
c = pd.DataFrame({'x': [5], 'z': [True]})
pd.concat([a, c], ignore_index=True)

,x,y,z
0,1,a,NaN
1,2,b,NaN
2,5,NaN,True


---
## 2. 실전: 고객정보 + 서비스정보 병합

In [12]:
cust = pd.read_csv('cust_info.csv')
svc = pd.read_csv('service_info.csv')
print(cust.shape, svc.shape)
display(cust.head(3), svc.head(3))

(7065, 7) (7043, 15)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,Churn
0,4157-ZYCBC,Female,0,Yes,Yes,60,No
1,1796-QPVMF,Female,1,Yes,No,32,No
2,4535-DTJDG,Female,0,No,Yes,15,No


,customerID,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,8650-YFFGJ,No,No phone service,Fiber optic,Yes,No,No,No,No,Yes,Month-to-month,Yes,Bank transfer (automatic),83.90,2331.69
1,4640-LCPSY,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,No,Mailed check,40.19,2647.74
2,1074-TDIUY,Yes,No,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Credit card (automatic),85.44,2132.81


### 2-1. 병합 전 점검: 키의 고유성과 중복

In [13]:
print("cust 고객ID 고유값 수:", cust['customerID'].nunique(), "/ 행 수:", len(cust))
print("svc  고객ID 고유값 수:", svc['customerID'].nunique(), "/ 행 수:", len(svc))
print("cust 완전 중복 행:", cust.duplicated().sum())

cust 고객ID 고유값 수: 7043 / 행 수: 7065
svc  고객ID 고유값 수: 7043 / 행 수: 7043
cust 완전 중복 행: 22


In [14]:
# 키가 중복된 채로 merge하면 행이 불어납니다 → 먼저 중복 제거
cust = cust.drop_duplicates().reset_index(drop=True)
print(cust.shape, cust['customerID'].is_unique)

(7043, 7) True


### 2-2. merge 수행

In [15]:
df = pd.merge(cust, svc, on='customerID', how='inner')
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,Churn,PhoneService,MultipleLines,InternetService,...,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,4157-ZYCBC,Female,0,Yes,Yes,60,No,Yes,Yes,DSL,...,No,Yes,Yes,Yes,No,Month-to-month,Yes,Credit card (automatic),92.73,5804.72
1,1796-QPVMF,Female,1,Yes,No,32,No,Yes,No,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Credit card (automatic),37.04,1182.75
2,4535-DTJDG,Female,0,No,Yes,15,No,Yes,Yes,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Credit card (automatic),45.80,712.06
3,9333-FUAHO,Female,0,No,No,41,Yes,Yes,Yes,Fiber optic,...,Yes,Yes,No,No,No,Month-to-month,Yes,Credit card (automatic),103.99,4329.13
4,5470-KGMYS,Female,0,Yes,NaN,28,Yes,Yes,Yes,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),42.80,1154.56


In [16]:
# 병합 결과 확인: left 병합과 행 수 비교, indicator로 매칭 여부 확인
chk = pd.merge(cust, svc, on='customerID', how='outer', indicator=True)
chk['_merge'].value_counts()

_merge
both          7043
left_only        0
right_only       0
Name: count, dtype: int64

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7013 non-null   object 
 5   tenure            7043 non-null   int64  
 6   Churn             7043 non-null   object 
 7   PhoneService      7043 non-null   object 
 8   MultipleLines     7043 non-null   object 
 9   InternetService   7043 non-null   object 
 10  OnlineSecurity    7043 non-null   object 
 11  OnlineBackup      7043 non-null   object 
 12  DeviceProtection  7043 non-null   object 
 13  TechSupport       7043 non-null   object 
 14  StreamingTV       7043 non-null   object 
 15  StreamingMovies   7043 non-null   object 
 16  Contract          7043 non-null   object 


> 📌 `TotalCharges`가 숫자처럼 보이는데 `object`로 되어 있습니다. 공백(`' '`) 값이 섞여 있기 때문입니다.
> 이런 문제는 14번 사례실습(통신사 이탈 예측)에서 본격적으로 처리합니다.

In [18]:
print((df['TotalCharges'] == ' ').sum())
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].dtype

11


dtype('float64')

### 2-3. 병합 데이터로 간단 분석

In [19]:
pd.crosstab(df['Contract'], df['Churn'], normalize='index').round(3)

Churn,No,Yes
Contract,,
Month-to-month,0.447,0.553
One year,0.827,0.173
Two year,0.958,0.042


In [20]:
df.groupby('InternetService')['MonthlyCharges'].mean().round(2)

InternetService
DSL             77.84
Fiber optic    102.63
No              40.16
Name: MonthlyCharges, dtype: float64

---
## 3. 실전: 같은 구조의 데이터 이어 붙이기 (concat)

In [21]:
d0 = pd.read_csv('cust_data.csv')
d1 = pd.read_csv('cust_data1.csv')
print(d0.shape, d1.shape, (d0.columns == d1.columns).all())

(9930, 10) (9930, 10) True


In [22]:
# 어느 파일에서 왔는지 표시하는 컬럼을 추가한 뒤 이어 붙이기
d0['src'] = 'cust_data'
d1['src'] = 'cust_data1'
both = pd.concat([d0, d1], ignore_index=True)
print(both.shape)
both.groupby('src').size()

(19860, 11)


src
cust_data     9930
cust_data1    9930
dtype: int64

In [23]:
# keys 옵션으로 멀티 인덱스 만들기
pd.concat([d0.head(2), d1.head(2)], keys=['d0', 'd1'])

class sex   age  service stop npay    avg_bill      A_bill     B_bill  \
d0 0     B   F  61.0        2    N    N   7190.9881   3479.1102  2878.7129   
   1     E   M  55.0        1    N    Y  25045.0340  23940.8983   648.1402   
d1 0     B   F  61.0        2    N    N   7190.9881   3479.1102  2878.7129   
   1     E   M  55.0        1    N    Y  25045.0340  23940.8983   648.1402   

     termination         src  
d0 0           N   cust_data  
   1           N   cust_data  
d1 0           N  cust_data1  
   1           N  cust_data1

---
## 4. 기타 결합 방법

In [24]:
# join: 인덱스 기준 결합
l = left.set_index('id')
r = right.set_index('id')
l.join(r, how='left')

,name,plan
id,,
1,kim,NaN
2,lee,5G
3,park,LTE
4,choi,5G


In [25]:
# 그룹 집계 결과를 원본에 붙이기 (자주 쓰는 패턴)
avg_by_contract = df.groupby('Contract', as_index=False)['MonthlyCharges'].mean().rename(columns={'MonthlyCharges': 'contract_avg'})
df2 = df.merge(avg_by_contract, on='Contract', how='left')
df2[['Contract', 'MonthlyCharges', 'contract_avg']].head()

,Contract,MonthlyCharges,contract_avg
0,Month-to-month,92.73,80.559232
1,One year,37.04,80.471299
2,Month-to-month,45.80,80.559232
3,Month-to-month,103.99,80.559232
4,Month-to-month,42.80,80.559232


---
## ✏️ 연습문제

**문제 1.** `cust_info.csv`와 `service_info.csv`를 읽어 각각 `ci`, `si`에 저장하고, `ci`의 중복 행을 제거하세요. 제거된 행의 개수를 `답안1` 변수에 저장하세요.

In [26]:
# 여기에 답안코드를 작성하고 실행하세요.

**문제 2.** `ci`를 기준으로(`left`) `si`를 `customerID`로 병합하여 `merged` 변수에 저장하고, 행과 열의 개수를 출력하세요.

In [27]:
# 여기에 답안코드를 작성하고 실행하세요.

**문제 3.** 아래 코드의 빈칸을 채워 `cust_data.csv`와 `cust_data1.csv`를 위아래로 붙이고 인덱스를 새로 매기세요.
```python
total = pd.< blank3-1 >([d0, d1], axis=0, < blank3-2 >=True)
```
빈칸 내용을 `답안3_1`, `답안3_2` 변수에 저장하세요.

In [28]:
# 아래 코드의 빈칸을 채우고 실행하세요.

**문제 4.** 아래 코드의 오류를 정정하세요. (계약형태 `Contract` 별 해지 고객 수를 구해 원본에 `churn_cnt` 컬럼으로 붙이기)
```python
cnt = merged[merged['Churn'] == 'Yes'].groupby('Contract').size().reset_index(name='churn_cnt')
result = pd.merge(merged, cnt, how='left')
result = pd.concat(result, cnt, axis=1)
```

In [29]:
# 아래의 코드를 정정하고 실행하세요.

---
## ✅ 정답 및 해설

In [30]:
# 문제 1
ci = pd.read_csv('cust_info.csv')
si = pd.read_csv('service_info.csv')
before = len(ci)
ci = ci.drop_duplicates()
답안1 = before - len(ci)
print(답안1)

# 문제 2
merged = pd.merge(ci, si, on='customerID', how='left')
print(merged.shape)

22
(7043, 21)


In [31]:
# 문제 3
d0 = pd.read_csv('cust_data.csv')
d1 = pd.read_csv('cust_data1.csv')
total = pd.concat([d0, d1], axis=0, ignore_index=True)
답안3_1 = 'concat'
답안3_2 = 'ignore_index'
print(total.shape, total.index[-1])

(19860, 10) 19859


In [32]:
# 문제 4 : merge에 on='Contract' 지정(명시가 안전), 세 번째 줄(concat)은 불필요하고 사용법도 틀림(리스트로 전달해야 함)
cnt = merged[merged['Churn'] == 'Yes'].groupby('Contract').size().reset_index(name='churn_cnt')
result = pd.merge(merged, cnt, on='Contract', how='left')
result[['Contract', 'churn_cnt']].drop_duplicates()

,Contract,churn_cnt
0,Month-to-month,2159
1,One year,249
5,Two year,71
